<a href="https://colab.research.google.com/github/lifan149/notes/blob/main/fastai/Practical-Deep-Learning-for-Coders/dlfc_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

根据[fastbook第四节](https://github.com/fastai/fastbook/blob/master/04_mnist_basics.ipynb)记录的笔记

In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

根据教程尝试创建一个可以将任何图像分类为 3 或 7 的模型。下载一个仅包含这些数字图像的 MNIST 示例

In [ ]:
path = untar_data(URLs.MNIST_SAMPLE)

In [ ]:
#hide
Path.BASE_PATH = path

通过 fastai 添加的方法 ls 来查看这个目录中的内容。此方法返回一个名为 L 的特殊 fastai 类的对象，该类具有与 Python 内置列表相同的功能，以及更多功能。它的一个方便的功能是，在打印时，它会在列出项目本身之前显示项目计数（如果超过 10 个项目，它只显示前几个项目）：

In [ ]:
path.ls()

MNIST 数据集遵循机器学习数据集的常见布局：训练集和验证集（和/或测试集）的单独文件夹。

In [ ]:
# 查看训练集的内容
(path/'train').ls()

有一个 3 的文件夹和一个 7 的文件夹。在机器学习术语中，我们说“3”和“7”是此数据集中的标签 （或目标）

In [ ]:
threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()
threes

一张手写数字 3 的图片，取自手写数字 MNIST 数据集

In [ ]:
im3_path = threes[1]
im3 = Image.open(im3_path)
im3

在计算机中，一切都表示为数字。要查看构成此图像的数字，我们必须将其转换为 NumPy 数组或 PyTorch 张量 。

**本质上，NumPy 数组（ndarray）和 PyTorch 张量（Tensor）在核心结构上都是多维数组**

**PyTorch 张量在 NumPy 数组的基础上，额外提供了 GPU 加速和自动求导的功能，使其更适合深度学习任务。**

将图像的一部分转为NumPy 数组

4:10 (对于行): 选取从索引为 4 的行开始，直到但不包括索引为 10 的行。换句话说，它会选择索引为 4, 5, 6, 7, 8, 9 的这 6 行。

4:10 (对于列): 这表示选取从索引为 4 的列开始，直到但不包括索引为 10 的列。同样地，它会选择索引为 4, 5, 6, 7, 8, 9 的这 6 列。

NumPy 从上到下、从左到右编制索引，因此此部分位于图像的左上角

In [ ]:
array(im3)[4:10,4:10]

 PyTorch 张量也是如此

In [ ]:
tensor(im3)[4:10,4:10]

## 关于图像数据的多维数组
---

### **1. 彩色图像的三维数组结构**
- **维度定义**  
  彩色图像通常表示为 **三维数组**，结构为 `(高度, 宽度, 颜色通道)`，例如 RGB 图像的通道数为 3（红、绿、蓝）。  
  - **示例**：一个分辨率为 `1920x1080` 的 RGB 图像，其数组形状为 `(1080, 1920, 3)`。  
  - **通道顺序**：OpenCV 中默认通道顺序为 **BGR**（蓝、绿、红），而非 RGB。

- **像素值含义**  
  每个通道的像素值范围通常为 **0-255**，表示颜色强度：  
  - `0`：表示该通道无贡献（如红色通道为 0 时，无红色成分）。  
  - `255`：表示该通道最大强度。

---

### **2. 灰度图像的二维数组结构**
- **维度定义**  
  灰度图像仅包含亮度信息，因此简化为 **二维数组**，结构为 `(高度, 宽度)`。  
  - **示例**：同样分辨率 `1920x1080` 的灰度图像，数组形状为 `(1080, 1920)`。

- **像素值含义**  
  每个像素值直接表示亮度，范围仍为 **0-255**：  
  - `0`：纯黑色。  
  - `255`：纯白色，中间值表示不同灰度等级。

---

### **3. 内存存储与数组组织**
- **多维数组的线性化**  
  尽管逻辑上是多维结构，但计算机内存是线性地址空间。多维数组通过 **行优先顺序** 存储，例如：  
  - 二维数组 `B[a][b]` 的内存布局为连续的 `a*b` 个元素。  
  - 三维数组（如 RGB 图像）则按通道顺序依次存储每个像素的红、绿、蓝值。

---

### **4. 应用场景对比**
- **彩色图像**  
  适用于需要颜色信息的场景（如物体识别、视频处理），但数据量较大（3 倍于灰度图像）。  
- **灰度图像**  
  常用于简化计算（如边缘检测、二值化处理），或颜色无关的分析任务。

---

### **示例代码（Python + OpenCV）**
```python
import cv2

# 读取彩色图像（三维数组）
color_img = cv2.imread("image.jpg")  # 形状为 (H, W, 3)，通道顺序为 BGR
print(color_img.shape)  # 输出：(高度, 宽度, 3)

# 转换为灰度图像（二维数组）
gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)  # 形状变为 (H, W)
print(gray_img.shape)  # 输出：(高度, 宽度)
```

---

**总结**：  
- 彩色图像通过三维数组表示颜色通道，灰度图像通过二维数组表示亮度。  
- 像素值范围 0-255 是数字图像的通用标准，与硬件（如显示器、传感器）的 8 位分辨率相关。


> 如果图像只包含 **0 和 255** 的像素值，则该图像属于 **二值图像**（Binary Image）。其特点是每个像素仅用两种值表示：  
- **0**：通常表示黑色（背景）。  
- **255**：通常表示白色（前景或目标物体）。  
>
> ---
>
> ### **关键特性与用途**
1. **图像分割**：通过阈值处理将灰度图像转换为二值图像，用于区分目标与背景（如边缘检测）。  
2. **简化计算**：仅保留关键信息，减少数据量，常用于OCR、条形码识别等场景。  
3. **形态学操作**：如腐蚀、膨胀等处理通常基于二值图像实现。  
>
> ---
>
> ### **与其他图像类型的对比**
- **灰度图像**：像素值范围为 **0-255**（连续灰度），而二值图像是灰度图像的特例。  
- **彩色图像**：需要三个通道（如RGB）存储颜色信息，而二值图像为单通道。  


对数组进行切片，只选择数字顶部的部分，然后使用 Pandas DataFrame 通过渐变对值进行颜色编码，这清楚地向我们展示了图像是如何从像素值创建的：

In [ ]:
#hide_output
im3_t = tensor(im3)
df = pd.DataFrame(im3_t[4:15,4:22])
df.style.set_properties(**{'font-size':'6pt'}).background_gradient('Greys')

可以看到，背景白色像素存储为数字 0，黑色像素存储为数字 255，灰色阴影介于两者之间。整个图像包含 28 个横向像素和 28 个向下像素，总共 784 个像素

**计算机如何识别3和7呢**

尝试使用像素相似度。找到 3/7 的每个像素的平均像素值，将得到两组平均值，定义我们可以称之为“理想”的 3 和 7。然后，要将图像分类为一个数字或另一个数字，我们可以看到图像与这两个理想数字中的哪一个最相似，这是一个很好的基线。

> 在机器学习和数据科学领域，**"baseline" (基线)** 指的是一个**简单、容易实现但性能可能不是最优的模型或方法**，用于作为比较的基准。它的作用是：
>>
* **提供一个起步点：** 在尝试更复杂、更精密的模型之前，先建立一个简单的基线模型，了解最基本的方法能够达到的性能水平。
* **衡量改进：** 后续开发的更复杂的模型或技术，其性能应该显著优于这个基线模型，才能证明其有效性。如果一个复杂的模型甚至不如简单的基线模型，那说明这个复杂模型可能存在问题。
* **理解问题的难度：** 基线模型的性能可以帮助我们初步了解分类任务的难易程度。如果一个非常简单的基线模型就能达到很高的准确率，那可能说明这个问题本身就比较容易。
>
> 总而言之，这里的 "baseline" 指的是一个简单但合理的起始方法，用于建立一个初步的分类性能水平，并作为未来更复杂方法进行比较的参考。

使用 Python 列表推导式来创建图像张量的普通列表

> 注意：列表推导式：列表和字典推导式是 Python 的一个很棒的功能。许多 Python 程序员每天都在使用它们，包括本书的作者 — 它们是“惯用 Python”的一部分。但是来自其他语言的程序员可能以前从未见过它们。只需在网上搜索一下，就有很多很棒的教程，所以我们现在不会花很长时间讨论它们。下面是一个快速说明和示例，可帮助您入门。列表推导式如下所示： new_list = [f(o) for o in a_list if o>0] .这将返回 a_list 中大于 0 的所有元素，然后将其传递给函数 f。这里有三个部分：你要迭代的集合 （a_list）、一个可选的过滤器 （if o>0） 和对每个元素执行的作 （f（o））。它不仅编写时间更短，而且比使用循环创建相同列表的替代方法要快得多。

In [ ]:
seven_tensors = [tensor(Image.open(o)) for o in sevens]
three_tensors = [tensor(Image.open(o)) for o in threes]
len(three_tensors),len(seven_tensors)

seven_tensors 和three_tensors 实际上是列表，图像张量的列表，打印出的结构虽然看上去和三维数组打印的结构一样，但是它们不是三维数组

使用 fastai 的 show_image 函数来显示它

In [ ]:
show_image(three_tensors[1]);

对于每个像素位置，我们想要计算该像素强度的所有图像的平均值。为此，我们首先将此列表中的所有图像组合成一个三维张量。描述此类张量的最常见方式是将其称为 rank-3 张量。

可以将二维张量相信成一张纸，所谓合成三维张量就像把多张形状相同的纸（二维张量）一层一层地堆叠起来形成一个长方体。

PyTorch 中的某些作（例如取平均值）需要我们将整数类型转换为浮点类型。由于我们稍后会需要它，因此我们现在还将 stacked tensor 转换为 float。在 PyTorch 中进行强制转换非常简单，只需键入要强制转换的类型的名称，并将其视为一种方法即可。

通常，当图像为浮点数时，像素值预期在 0 和 1 之间，因此我们在这里也要除以 255：

In [ ]:
stacked_sevens = torch.stack(seven_tensors).float()/255
stacked_threes = torch.stack(three_tensors).float()/255
stacked_threes.shape

根据上面执行结果 (6131, 28, 28)

它的形状是 (6131, 28, 28)。这个形状告诉我们：
* 第一个轴（维度）的长度是 6131。这通常表示我们有 6131 个样本，在这个上下文中是 6131 张图像。
* 第二个轴的长度是 28。这通常表示每张图像的高度是 28 像素。
* 第三个轴的长度是 28。这通常表示每张图像的宽度是 28 像素。

对于 PyTorch 来说，一个形状为 (6131, 28, 28) 的张量仅仅是在内存中存储的一串数字，按照这个特定的多维结构排列。**张量的形状只是数据的组织结构，而每个维度代表什么含义，是由我们根据实际应用场景来解释的**


张量形状的长度

In [ ]:
len(stacked_threes.shape)

张量的**秩就是张量的维度数量，也就是 `shape` 元组的长度。**

* 对于一个标量（例如 `5`），其 `shape` 是 `()`（空元组），因此秩是 0。
* 对于一个向量（例如 `[1, 2, 3]`），其 `shape` 是 `(3,)`，因此秩是 1。
* 对于一个矩阵（例如 `[[1, 2], [3, 4]]`），其 `shape` 是 `(2, 2)`，因此秩是 2。
* 对于我们例子中的图像张量，其 `shape` 是 `(6131, 28, 28)`，因此秩是 3（它是一个 rank-3 张量或三维张量）。

**理解张量的两个基本属性：**
1.  **形状 (shape):** 描述了张量在每个维度上的大小，是理解数据组织方式的关键。
2.  **秩 (rank):** 描述了张量的维度数量。


**rank 是张量中的轴或维度的数量;shape 是张量每个轴的大小**

---

**注意："维度"（dimension）在不同语境下的不同含义**

1. **物理空间中的“维度” vs. 张量的“秩”（rank）**
* 物理空间的三维性：当我们说“三维空间”时，是指描述一个点的位置需要 三个独立参数（例如 x, y, z 坐标）。这里的“维度”指的是 空间方向的数量，每个方向对应一个轴（axis），且每个轴的长度（元素数量）可能不同。

* 张量的秩（rank）：在 PyTorch 中，`v.ndim` 返回张量的 秩（即轴的个数），而非每个轴的长度。例如：

* 一个三维空间位置向量 `v = [1, 2, 3]` 在 PyTorch 中是 一维张量，因为它的 `ndim=1`，仅有一个轴，该轴的长度为 3（`shape=[3]`）。

* 矩阵（二维张量）的 `ndim=2`，因为它有两个轴（行和列）。


> 关键区分：  
> - 物理中的“三维”对应张量的 某个轴的长度（如 `shape=[3]`）；  
> - 张量的“维度”（`ndim`）实际是 秩（轴的个数）。

2. **术语歧义的根源**

* “维度”一词的多义性：

  * 轴的数量（Rank）：在张量中，`ndim` 表示秩（如矩阵是二维张量）。

  * 轴的长度（Size/Shape）：在物理中，“三维”指每个轴的长度（如 `shape=[3]`）。

* 混淆示例：

  * 若有人说“三维张量”，可能指：

    1. 秩为 3 的张量（如 `shape=[2,3,4]`，三个轴）；
    2. 物理空间中某个轴的长度为 3 的张量（如 `shape=[3]`，秩为 1）。

3. **如何避免混淆？**
建议使用无歧义的术语：
* 秩（Rank）：张量的轴数（`ndim`）。

* 轴（Axis）：张量的某个独立方向（如矩阵的行或列）。

* 长度（Shape）：每个轴上的元素数量（如 `shape=[3,4]` 表示第一轴长 3，第二轴长 4）。


举例说明：
* 物理位置向量 `v = [x, y, z]` 在 PyTorch 中：

* 秩为 1（一维张量）；

* 轴长度为 3（`shape=[3]`）；

* 它的“三维性”体现在轴的长度上，而非张量的秩。


4. **总结：理解张量的核心属性**
* `ndim`（秩）：轴的个数（如向量=1，矩阵=2）。

* `shape`（形状）：每个轴的长度（如 `shape=[3,4]` 表示二维张量，第一轴长 3，第二轴长 4）。

* 物理意义的“维度”：通常对应 `shape` 中的某个值，而非 `ndim`。


> 实际应用：在 PyTorch 中，讨论张量时应明确使用 秩、轴、形状，而非物理中的“维度”概念，以避免误解。

也可以使用 ndim 获取张量的秩：

In [ ]:
stacked_threes.ndim

对于每个像素位置，这将计算该像素在所有图像上的平均值。结果将是每个像素位置一个值，或单个图像

In [ ]:
mean3 = stacked_threes.mean(0)
show_image(mean3);

PyTorch张量的 mean 方法表示沿指定维度求均值

* **秩 (Rank)**：指的是张量的**轴的数量**，也就是 `shape` 元组的长度。对于 `stacked_threes`，如果它的形状是 `(num_images, height, width)`，那么它的秩是 **3**。

* **维度 (Dimension) 的索引**：当我们说“维度 0”、“维度 1”、“维度 2” 时，我们是在指张量的**不同轴**。

* **维度的大小 (Length of the axis)**：每个维度都有其大小。在 `stacked_threes` 中：
    * 维度 0 的大小是 `num_images`。
    * 维度 1 的大小是 `height`。
    * 维度 2 的大小是 `width`。

**所以，`stacked_threes` 的秩是 3，表示它有三个轴（维度）。**

你可能会想用“秩”来指代 `num_images`，但更准确的说法是：

* **维度 0 的大小** 表示图像的数量。
* **沿着秩为 0 的轴** 进行 `mean()` 操作，会对所有图像的对应像素取平均。

**为了避免歧义，我们可以这样说：**

在 `stacked_threes` 张量中（假设形状为 `(num_images, height, width)`）：

* **维度 0** 是图像数量的维度，其大小是 `num_images`。
* **维度 1** 是高度的维度，其大小是 `height`。
* **维度 2** 是宽度的维度，其大小是 `width`。

当我们对 `stacked_threes` 使用 `mean(0)` 时，我们是在**沿着图像数量这个维度**进行平均。

**总结：**

* **秩**是轴的数量。
* **维度索引**用于指代特定的轴。
* 每个维度都有其**大小**。

关于mean的参数，对于一个秩为 n 的张量，参数的取值范围是 ​​[-n, n-1]​​，允许正数和负数索引

比如使用 `stacked_threes` 的形状 `(num_images, height, width)` 来解释：

* **正向索引：**
    * 维度 `0` 是第一个维度（图像数量）。
    * 维度 `1` 是第二个维度（高度）。
    * 维度 `2` 是第三个维度（宽度）。

* **负向索引：**
    * 维度 `-1` 是**最后一个**维度（宽度）。
    * 维度 `-2` 是**倒数第二个**维度（高度）。
    * 维度 `-3` 是**倒数第三个**维度（图像数量）。

**所以，负数索引并不是从“不同的方向开始计算”平均值本身，而是提供了一种不同的方式来**指定**你想要进行平均操作的维度。** 计算平均值的过程（对该维度上的所有元素取平均）是相同的，只是你指定维度的**方式**不同。

对 7 做同样的事情

In [ ]:
mean7 = stacked_sevens.mean(0)
show_image(mean7);

In [ ]:
a_3 = stacked_threes[1]
show_image(a_3);


当需要量化图像与理想目标（如数字 "3"）的差异时，直接对像素差异求和会产生误导：正负差异相互抵消（例如某区域过暗补偿另一区域过亮），导致总差异看似为零。为解决此问题，需采用两种数学方法消除符号干扰。

---

**方法一：L1 范数（Mean Absolute Difference）**
1. **计算原理**
* **数学公式**：  

  $ L1 = \frac{1}{n} \sum_{i=1}^{n} |x_i - y_i| $

  其中 \(x_i\) 是实际像素值，\(y_i\) 是理想像素值，\(n\) 是像素总数。

* **关键操作**：  对每个像素的差异取绝对值后求平均，消除正负抵消效应。

2. **技术特性**

* **对异常值的鲁棒性**：  

  L1 对单个大误差的敏感度较低（绝对值线性增长），适用于存在噪声或离群点的场景。例如，若图像局部存在异常光斑，L1 的误差增幅较小。

* **稀疏性优势**：  

  在优化过程中，L1 倾向于产生稀疏解（部分差异归零），有助于识别关键差异区域。

* **计算效率**：  

  无需平方运算，计算速度较快，适用于实时系统。

3. **应用示例**  
假设理想数字 "3" 的某像素值为 200，实际图像中该像素值为 180，另一像素值为 220：  

* **直接差异**：\( (180-200) + (220-200) = -20 + 20 = 0 \)（错误结论）  

* **L1 差异**：\( |180-200| + |220-200| = 20 + 20 = 40 \)（正确反映总差异）


---

**方法二：L2 范数（Root Mean Squared Error, RMSE）**
1. **计算原理**
* **数学公式**：  

  $ L2 = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - y_i)^2} $
  
  先对差异平方以消除符号，再求均方根。

2. **技术特性**
* **对大差异的敏感性**：  

  平方运算放大较大误差的影响，使模型更关注显著偏离目标的情况。例如，若某像素差异为 50，其对 L2 的贡献为 2500，远高于 L1 的 50。
* **平滑优化特性**：  

  平方函数处处可导，利于梯度下降等优化算法稳定收敛。
* **物理意义**：  

  RMSE 的单位与原始数据一致（因平方根操作），便于直观解释。

3. **应用示例**  
同一场景下：  
* **L2 差异**：

$ \sqrt{(180-200)^2 + (220-200)^2} = \sqrt{400 + 400} ≈ 28.28 $

  相比 L1 的 40，L2 更强调单个像素的偏离程度。

---

**方法对比与选择建议**

| 维度         | L1 范数                          | L2 范数                          |
|-------------------|--------------------------------------|--------------------------------------|
| 误差分布敏感性   | 对异常值不敏感                       | 对大误差敏感                         |
| 优化特性       | 可能收敛至多个局部最优解             | 单一全局最优解                       |
| 计算复杂度     | 低（无需平方运算）                   | 略高（需平方与开方）                 |
| 典型应用场景   | 图像去噪、特征选择               | 高精度配准、质量评估             |


**选择策略**：
* **优先 L1**：若差异分析需排除噪声干扰（如医学图像中的伪影）或强调关键区域。

* **优先 L2**：若需精确量化整体偏离程度（如卫星图像配准）或依赖梯度优化。


---

**扩展：其他相关度量**
1. **Huber Loss**：  
   结合 L1 和 L2 的优点，对小误差使用平方项，对大误差切换为线性项，平衡鲁棒性与敏感性。
2. **结构相似性（SSIM）**：  
   超越像素级差异，从亮度、对比度、结构三方面评估图像相似性，适用于人类视觉感知优化。

---

通过合理选择 L1 或 L2 范数，可有效解决像素差异抵消问题，并为图像质量评估、目标识别等任务提供量化依据。实际应用中常结合交叉验证确定最优方法。

**鲁棒性（Robustness）的定义**

鲁棒性（Robustness）指系统、模型或算法在面对数据扰动、噪声、异常值或分布变化时，仍能保持稳定性和可靠性的能力。在统计学和机器学习中，对异常值的鲁棒性特指模型或方法对数据中极端值（异常值）的敏感度较低，即使存在少量异常值，也能给出准确且一致的结果。

---

**鲁棒性的核心思想**
1. **抗干扰能力**

   鲁棒性强调在数据存在噪声、缺失值或异常值时，模型不会因这些干扰而崩溃或产生严重偏差。例如，使用中位数（而非均值）作为位置估计量时，对极端值的影响更小。  
   * 示例：若数据集包含少数极高薪资（异常值），用均值计算平均薪资会被拉高，而中位数则能更稳健地反映典型薪资水平。


2. **分布适应性**  
   鲁棒方法不依赖严格的数据分布假设（如正态分布），能够适应非对称、偏态或未知分布的数据。  
   * 示例：鲁棒回归模型（如Huber回归）在存在异常值时，仍能拟合出合理的回归线，而普通线性回归可能被异常值严重干扰。


3. **稳定性与泛化性**

   鲁棒模型在训练数据与测试数据分布不同（分布偏移）时，仍能保持较好的性能。例如，在金融欺诈检测中，即使欺诈模式随时间变化，鲁棒模型仍能有效识别新类型的异常交易。

---

**对异常值的鲁棒性具体表现**
1. **异常值敏感性低**  
   * 传统方法问题：如均值和标准差对异常值敏感，一个极端值可能使结果严重偏离真实情况。  

   * 鲁棒替代方法：  

     * 用中位数代替均值，用MAD（Median Absolute Deviation）代替标准差。  

     * 使用L1范数（MAE）而非L2范数（MSE）作为损失函数，减少大误差的权重。


2. **异常值处理机制**  
   * **检测与剔除**：通过统计方法（如箱线图、Z-score）或机器学习方法（如孤立森林、LOF算法）识别异常值，再决定保留、修正或剔除。  

   * **建模兼容性**：鲁棒模型（如随机森林、支持向量机）在训练时自动降低异常值的影响，而非依赖预处理。


3. **结果可靠性**  
   * 即使数据包含一定比例的异常值，鲁棒方法的输出结果仍接近真实值。例如，在图像识别中，鲁棒模型对输入图像的噪声或遮挡具有更强的容忍度。


---

**鲁棒性的应用场景**
1. **数据分析与统计建模**  
   * **金融领域**：处理市场波动中的极端交易数据。  

   * **医疗领域**：识别罕见疾病数据，避免误诊。


2. **机器学习与深度学习**  
   * **异常检测**：如信用卡欺诈检测、工业设备故障预警。  

   * **模型优化**：通过对抗训练、正则化技术（如Dropout、权重约束）提升模型鲁棒性。


3. **工程与控制系统**
   * **自动驾驶**：在传感器噪声或环境干扰下稳定决策。  

   * **制造业**：在设备数据存在测量误差时，仍能准确预测故障。


---

**增强鲁棒性的方法**
1. **数据预处理**  
   * 清洗噪声数据、填充缺失值、标准化/归一化。  

   * 使用重采样技术（如Bootstrap）评估参数稳定性。


2. **算法选择**  
   * 优先选择非参数方法（如K近邻、决策树）或集成学习（如随机森林）。  

   * 采用鲁棒损失函数（如Huber损失、分位数损失）。


3. **模型设计**  
   * 引入正则化（L1/L2正则化）防止过拟合。  

   * 对抗训练：生成对抗样本并加入训练集，提升模型对扰动的抵抗力。


---

**总结**

鲁棒性是现代数据科学和机器学习的核心要求之一，尤其在现实数据普遍存在噪声和异常值的场景下。通过选择鲁棒方法、优化数据流程和模型设计，可显著提升系统在复杂环境中的可靠性。如需进一步了解具体算法（如MAD计算、LOF异常检测）或应用案例，可参考相关文献或技术文档。

In [ ]:
# 使用L1范数
dist_3_abs = (a_3 - mean3).abs().mean()
# 使用L2范数
dist_3_sqr = ((a_3 - mean3)**2).mean().sqrt()
dist_3_abs,dist_3_sqr

In [ ]:
dist_7_abs = (a_3 - mean7).abs().mean()
dist_7_sqr = ((a_3 - mean7)**2).mean().sqrt()
dist_7_abs,dist_7_sqr

在这两种情况下，我们的 3 和“理想”3 之间的距离都小于到理想 7 的距离。因此，在这种情况下，我们的简单模型将给出正确的预测。

1.  **PyTorch 提供内置的损失函数：** PyTorch 的 `torch.nn.functional` 模块（通常导入为 `F`）包含了许多常用的神经网络函数，包括我们讨论的平均绝对差（L1 范数）和均方根误差（RMSE 或 L2 范数）。

2.  **推荐的导入方式：** PyTorch 团队建议将 `torch.nn.functional` 模块导入为 `F`。fastai 库也遵循这个惯例，因此在 fastai 环境中，你可以直接使用 `F` 来访问这些函数。

3.  **计算 `a_3` 与 `mean7` 之间的距离：** 代码 `F.l1_loss(a_3.float(), mean7), F.mse_loss(a_3, mean7).sqrt()` 展示了如何使用这两个函数来计算一个数字 3 的图像张量 `a_3` 与“理想”数字 7 的图像张量 `mean7` 之间的距离。


In [ ]:
# mse_loss 代表均方误差 ，l1_loss 指的是平均绝对值的标准数学术语（在数学中称为 L1 范数 ）
# L1 范数和均方误差 （MSE） 之间的区别在于，后者会比前者更严厉地惩罚较大的错误（并且对小错误更宽容）
F.l1_loss(a_3.float(),mean7), F.mse_loss(a_3,mean7).sqrt()

## 激活函数和损失函数

**损失函数（Loss Function）**

**定义与作用**

损失函数是机器学习与深度学习中用于量化模型预测值与真实值之间差异的数学工具。它通过计算模型输出的误差，为参数优化提供方向，是模型训练的核心指标。

* **核心功能**：

  * **评估性能**：衡量模型预测的准确性，损失值越小表示模型越接近真实结果。

  * **指导优化**：作为优化算法（如梯度下降）的目标函数，通过最小化损失调整模型参数。

  * **正则化**：某些损失函数（如L1/L2）可加入正则项，防止过拟合。


**常见类型**

1. **回归任务**：
   * **L1损失（MAE）**：计算绝对误差，对异常值鲁棒，适用于数据噪声较大的场景。

   * **L2损失（MSE）**：计算平方误差，对离群点敏感，常用于精确回归。

   * **Huber损失**：结合L1和L2特性，对小误差平滑处理，对大误差线性处理，平衡鲁棒性与敏感性。


2. **分类任务**：
   * 交叉熵损失：衡量概率分布差异，多用于分类模型（如逻辑回归、神经网络）。

   * Hinge损失：支持向量机（SVM）的核心，通过最大化分类边界提升稀疏性。

   * 指数损失：AdaBoost算法的关键，对错误分类样本施加指数级惩罚，加速模型迭代。


**应用场景**
* **工业优化**：例如工厂通过损失函数计算最佳通风条件，以最小化额外成本。

* **模型训练**：如PyTorch中的`F.l1_loss`和`F.mse_loss`直接用于参数调优。


---

**激活函数（Activation Function）**

**定义与作用**

激活函数是神经网络中引入非线性的核心组件，决定神经元是否被激活及输出强度，使模型能够拟合复杂数据模式。

* **核心功能**：

  * **引入非线性**：打破线性限制，使网络能够学习复杂函数（如图像识别中的边缘检测）。

  * **梯度控制**：防止梯度消失或爆炸，例如ReLU在正区间保持梯度稳定。

  * **输出规范化**：将输出限制到特定范围（如Sigmoid映射到0-1，适合概率输出）。


**常见类型**
1. **Sigmoid**：输出0-1区间，适用于二分类的概率输出，但易导致梯度消失。
2. **Tanh**：输出-1到1区间，零中心化特性有利于优化过程，但仍存在梯度问题。
3. **ReLU**：正区间梯度为1，计算高效，广泛用于隐藏层，但可能导致“死亡神经元”。
4. **LeakyReLU/ELU**：改进ReLU的负区间处理，缓解神经元死亡问题。
5. **Softmax**：多分类任务中归一化输出为概率分布。

**应用场景**
* **隐藏层**：通常选择ReLU或其变体以提升训练效率。

* **输出层**：根据任务选择激活函数（如二分类用Sigmoid，多分类用Softmax）。


---

**区别与联系**

| 维度       | 损失函数                     | 激活函数                     |
|----------------|----------------------------------|----------------------------------|
| 功能定位   | 全局模型性能评估与优化目标       | 局部神经元非线性转换              |
| 数学作用   | 计算预测值与真实值的误差         | 决定神经元输出值                  |
| 典型示例   | 交叉熵、MSE                     | ReLU、Sigmoid                   |
| 优化关联   | 直接指导参数更新方向             | 影响反向传播梯度计算              |
| 任务适配性 | 需匹配任务类型（分类/回归）      | 需匹配网络层需求（隐藏层/输出层） |

**协同作用**

* 在神经网络中，激活函数处理单层非线性，而损失函数综合各层误差指导整体优化。例如，交叉熵损失需搭配Softmax激活函数实现多分类概率输出。


---

**总结**

损失函数和激活函数是深度学习的核心组件。前者作为“导航仪”评估和优化模型，后者作为“引擎”赋予网络非线性能力。合理选择两者（如回归任务用MSE+ReLU，分类任务用交叉熵+Softmax）可显著提升模型性能。更多细节可参考[PyTorch文档](https://pytorch.org/docs/stable/nn.html#loss-functions)或相关论文。

### NumPy 数组 和 PyTorch 张量


**1. NumPy 与 PyTorch 的共性**

两者均提供多维数组操作的 API，支持高效的数学运算（如矩阵乘法、广播机制等）。例如，以下代码在两种库中行为相似：
```python
# NumPy
import numpy as np
arr = np.array([[1, 2], [3, 4]])
result = arr * 2  # 输出 [[2,4], [6,8]]

# PyTorch
import torch
tensor = torch.tensor([[1, 2], [3, 4]])
result = tensor * 2  # 输出 tensor([[2,4], [6,8]])
```

---

**2. 为何深度学习选择 PyTorch 张量？**

尽管 API 相似，PyTorch 张量在以下关键特性上超越 NumPy 数组：

**（1）GPU 加速支持**

* **PyTorch张量**可通过 `.to("cuda")` 移动到 GPU 上计算，利用 CUDA 并行架构加速大规模矩阵运算。例如：

  ```python
  if torch.cuda.is_available():
      tensor_gpu = tensor.cuda()  # 将张量移至 GPU
  ```
* NumPy：仅支持 CPU 计算，无法直接利用 GPU 的并行能力。


**（2）自动微分（Autograd）**

* **PyTorch**：通过 `requires_grad=True` 跟踪张量操作，使用 `.backward()` 自动计算梯度。例如：

  ```python
  x = torch.tensor(3.0, requires_grad=True)
  y = x**2 + 2*x
  y.backward()  # 自动计算 dy/dx = 8.0
  ```
* **NumPy**：无内置梯度计算功能，需手动实现反向传播。


**（3）动态计算图**

* **PyTorch**：支持动态图机制，允许在运行时根据条件分支或循环修改计算流程。例如：

  ```python
  def dynamic_model(x):
      if x.sum() > 0:
          return x * 2
      else:
          return x - 1
  ```
* **NumPy**：仅提供静态数组操作，无法构建动态计算图。


---

**3. 为何 NumPy 仍不可替代？**

**（1）通用科学计算**

NumPy 是 Python 科学计算生态的基石，与 Pandas、Matplotlib 等库深度集成，适合数据分析、信号处理等非深度学习场景。

**（2）轻量级与稳定性**

对于小规模数据或无需 GPU/梯度计算的场景，NumPy 的 CPU 实现更轻量且稳定。

**（3）协同工作**

两者可通过零拷贝转换无缝协作：
```python
# NumPy 转 Tensor（共享内存）
np_array = np.ones(5)
torch_tensor = torch.from_numpy(np_array)

# Tensor 转 NumPy（共享内存）
torch_tensor = torch.ones(5)
np_array = torch_tensor.numpy()
```

---

**4. 总结与选择建议**

| 维度       | NumPy 数组                  | PyTorch 张量                  |
|----------------|--------------------------------|-----------------------------------|
| 核心用途   | 通用科学计算、数据分析         | 深度学习模型开发与训练            |
| 硬件加速   | 仅 CPU                         | 支持 GPU/CUDA 加速               |
| 自动求导   | 不支持                         | 通过 `autograd` 支持              |
| 动态图     | 不支持                         | 支持运行时动态修改计算流程        |
| 内存管理   | 不可变数组，可能产生内存拷贝   | 视图机制优化内存共享              |

选择策略：
* **深度学习任务**：优先使用 PyTorch 张量，利用其 GPU 加速、自动微分和动态图特性。

* **传统科学计算**：使用 NumPy 数组，结合 Pandas、SciPy 等库完成数据处理。


---

**5. 扩展说明**

• **Fastai 的增强功能**：Fastai 库对 NumPy 和 PyTorch 进行了扩展，使两者的 API 更加接近。例如，用户可通过 `from fastai.vision.all import *` 引入兼容性支持。

• **性能优化**：PyTorch 张量在底层融合操作（如矩阵乘法 + ReLU 激活）以减少 GPU 内核调用次数，而 NumPy 无此类优化。